# Notebook 00 — Data Preparation and Stage-1 Classifiers (GPU)

Single-cell version for the GPU machine. Trains the TF-IDF+LR main backend plus three Korean PLMs (klue/roberta-base, klue/bert-base, KoELECTRA) and saves per-backend test predictions for the RQ1 robustness comparison in notebook 06.

In [1]:
# ============================================================================
# Notebook 00 — Data Preparation and Stage-1 Classifiers (GPU, 3 backends)
# Study A: Simulation-based Seller-Level SCS and Cost-Optimal Threshold
#
# Trains the TF-IDF+LR main backend (kept as the primary classifier, following
# the first paper's structure) plus three Korean PLMs — klue/roberta-base,
# klue/bert-base, and KoELECTRA — for RQ1 multi-backend robustness. Each
# backend's full test-set predictions (predicted segment + confidence) are
# saved so that notebooks 01-06 can be re-run per backend.
#
# First-paper hyperparameters (p.12-13): max_seq_len=64, AdamW,
# warmup_ratio=0.1, 5 epochs, batch=64, lr=2e-5.
# ============================================================================

import os
import json
import time
import warnings
import random
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---- Reproducibility -------------------------------------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only")

# ---- Paths (notebook runs from study_A/notebooks/) -------------------------
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(ROOT, "data")
ARTIFACT_DIR = os.path.join(ROOT, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ---- Load splits -----------------------------------------------------------
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
N_CLASSES = int(train_df["label"].nunique())
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {df.shape[0]:6d} rows, {df['label'].nunique()} classes")
print("Number of segment classes:", N_CLASSES)

# ===========================================================================
# PART 1 - TF-IDF + Logistic Regression (MAIN backend)
# Character n-gram TF-IDF captures Korean sub-word morphology without a
# tokenizer. Its test predictions drive notebooks 01-05 by default.
# ===========================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

main_clf = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5),
                              min_df=2, max_features=200000, sublinear_tf=True)),
    ("lr", LogisticRegression(C=10.0, max_iter=1000, n_jobs=-1,
                              random_state=SEED)),
])
t0 = time.time()
main_clf.fit(train_df["prod_name"], train_df["label"])
print(f"[TF-IDF+LR] trained in {time.time() - t0:.1f}s")

test_pred = main_clf.predict(test_df["prod_name"])
test_acc = accuracy_score(test_df["label"], test_pred)
test_f1 = f1_score(test_df["label"], test_pred, average="macro")
print(f"[TF-IDF+LR] test acc {test_acc:.4f}  macro-F1 {test_f1:.4f}")

test_proba = main_clf.predict_proba(test_df["prod_name"])
pred_df = test_df.copy().reset_index(drop=True)
pred_df["pred"] = test_pred
pred_df["confidence"] = test_proba.max(axis=1)
pred_df["correct"] = (pred_df["pred"] == pred_df["label"]).astype(int)

# MAIN predictions consumed by notebooks 01-05
np.save(os.path.join(ARTIFACT_DIR, "test_proba.npy"), test_proba)
pred_df.to_csv(os.path.join(ARTIFACT_DIR, "test_predictions.csv"), index=False)
# Backend-tagged copy for the RQ1 comparison in notebook 06
pred_df.to_csv(os.path.join(ARTIFACT_DIR, "test_predictions_tfidf_lr.csv"), index=False)

# ===========================================================================
# PART 2 - Three Korean PLMs (GPU) for RQ1 robustness
# ===========================================================================
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
import torch.nn.functional as F

PLM_MODELS = {
    "klue_roberta_base": "klue/roberta-base",
    "klue_bert_base":    "klue/bert-base",
    "koelectra_base":    "monologg/koelectra-base-v3-discriminator",
}

MAX_LEN = 64
NUM_EPOCHS = 5
BATCH_SIZE = 64
LR = 2e-5
WARMUP_RATIO = 0.1


def make_hf_dataset(df, tokenizer):
    ds = Dataset.from_pandas(df[["prod_name", "label"]].reset_index(drop=True))
    def tok(batch):
        return tokenizer(batch["prod_name"], truncation=True, max_length=MAX_LEN)
    ds = ds.map(tok, batched=True, remove_columns=["prod_name"])
    ds = ds.rename_column("label", "labels")
    return ds


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}


plm_summary = {}
for tag, model_name in PLM_MODELS.items():
    print(f"\n===== Fine-tuning {tag} ({model_name}) =====")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=N_CLASSES)

    train_ds = make_hf_dataset(train_df, tokenizer)
    val_ds = make_hf_dataset(val_df, tokenizer)
    test_ds = make_hf_dataset(test_df, tokenizer)
    collator = DataCollatorWithPadding(tokenizer)

    # TrainingArguments' evaluation-strategy arg was renamed across
    # transformers versions (evaluation_strategy -> eval_strategy). Pick the
    # name supported by the installed version so both work.
    import inspect
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"
    ta_kwargs = {
        "output_dir": os.path.join(ARTIFACT_DIR, f"_hf_{tag}"),
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": 128,
        "learning_rate": LR,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": 0.01,
        eval_key: "epoch",
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "logging_steps": 100,
        "fp16": (DEVICE == "cuda"),
        "seed": SEED,
        "report_to": "none",
        "save_total_limit": 1,
    }
    args = TrainingArguments(**ta_kwargs)
    # transformers >=4.46 renamed Trainer's "tokenizer" arg to
    # "processing_class"; branch so the code runs on both old and new versions.
    import inspect
    trainer_kwargs = dict(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer
    trainer = Trainer(**trainer_kwargs)
    t0 = time.time()
    trainer.train()
    train_secs = time.time() - t0
    print(f"[{tag}] trained in {train_secs/60:.1f} min")

    pred_out = trainer.predict(test_ds)
    logits = torch.tensor(pred_out.predictions)
    probs = F.softmax(logits, dim=-1).numpy()
    preds = probs.argmax(axis=1)
    conf = probs.max(axis=1)

    acc = accuracy_score(test_df["label"], preds)
    mf1 = f1_score(test_df["label"], preds, average="macro")
    print(f"[{tag}] TEST acc {acc:.4f}  macro-F1 {mf1:.4f}")

    bdf = test_df.copy().reset_index(drop=True)
    bdf["pred"] = preds
    bdf["confidence"] = conf
    bdf["correct"] = (bdf["pred"] == bdf["label"]).astype(int)
    bdf.to_csv(os.path.join(ARTIFACT_DIR, f"test_predictions_{tag}.csv"), index=False)
    np.save(os.path.join(ARTIFACT_DIR, f"test_proba_{tag}.npy"), probs)

    plm_summary[tag] = {"model": model_name,
                        "test_accuracy": round(float(acc), 4),
                        "test_macro_f1": round(float(mf1), 4),
                        "train_minutes": round(train_secs / 60, 1)}

    del model, trainer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# ===========================================================================
# PART 3 - Summary
# ===========================================================================
summary = {
    "n_classes": N_CLASSES,
    "train_size": int(train_df.shape[0]),
    "test_size": int(test_df.shape[0]),
    "main_backend": "tfidf_char_wb_2_5 + logreg",
    "main_test_accuracy": round(float(test_acc), 4),
    "main_test_macro_f1": round(float(test_f1), 4),
    "plm_backends": plm_summary,
}
with open(os.path.join(ARTIFACT_DIR, "nb00_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("\n===== FINAL SUMMARY =====")
print(json.dumps(summary, indent=2, ensure_ascii=False))


Torch: 2.6.0+cu124 | Device: cuda | NVIDIA GeForce RTX 4060 Ti
train:  76996 rows, 11 classes
val  :  10988 rows, 11 classes
test :  22011 rows, 11 classes
Number of segment classes: 11
[TF-IDF+LR] trained in 53.8s
[TF-IDF+LR] test acc 0.8639  macro-F1 0.8636

===== Fine-tuning klue_roberta_base (klue/roberta-base) =====


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 13133.87it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 22011/22011 [00:00<00:00, 58301.02 examples/s]
[transformers] warmup_ratio is deprecated and will be re

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.561634,0.532287,0.834820,0.836804
2,0.452752,0.485469,0.856389,0.853587
3,0.342436,0.453266,0.865763,0.865175
4,0.272780,0.441821,0.873225,0.872815
5,0.211311,0.461327,0.871496,0.871292


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.77it/s]


[klue_roberta_base] trained in 13.4 min


...

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 32835.71it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initial

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.703834,0.655450,0.802512,0.804032
2,0.543557,0.566837,0.836276,0.832339
3,0.421836,0.516741,0.848289,0.847725
4,0.371693,0.510548,0.852657,0.852491
5,0.317353,0.510248,0.855479,0.855293


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.73it/s]


[koelectra_base] trained in 13.0 min


[koelectra_base] TEST acc 0.8586  macro-F1 0.8585

===== FINAL SUMMARY =====
{
  "n_classes": 11,
  "train_size": 76996,
  "test_size": 22011,
  "main_backend": "tfidf_char_wb_2_5 + logreg",
  "main_test_accuracy": 0.8639,
  "main_test_macro_f1": 0.8636,
  "plm_backends": {
    "klue_roberta_base": {
      "model": "klue/roberta-base",
      "test_accuracy": 0.8728,
      "test_macro_f1": 0.8723,
      "train_minutes": 13.4
    },
    "klue_bert_base": {
      "model": "klue/bert-base",
      "test_accuracy": 0.863,
      "test_macro_f1": 0.8628,
      "train_minutes": 13.8
    },
    "koelectra_base": {
      "model": "monologg/koelectra-base-v3-discriminator",
      "test_accuracy": 0.8586,
      "test_macro_f1": 0.8585,
      "train_minutes": 13.0
    }
  }
}
